In [ ]:
# 安装 pyarrow（推荐，更稳定）
!pip install importlib -i https://pypi.tuna.tsinghua.edu.cn/simple

In [ ]:
import subprocess
import sys
import os
from pathlib import Path

files = ["CSAD 模型（基于板块中地位）.py"]

workdir = Path.cwd().resolve()

for filename in files:
    script_path = workdir / filename

    print(f"正在执行: {script_path}")
    print(f"使用解释器: {sys.executable}")

    run_env = os.environ.copy()
    run_env["PYTHONUNBUFFERED"] = "1"

    proc = subprocess.Popen(
        [
            sys.executable,  # 关键：使用当前 Notebook Kernel 的 Python
            "-u",            # 关闭输出缓冲
            str(script_path),
        ],
        cwd=str(workdir),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=run_env,
    )

    for line in proc.stdout:
        print(line, end="")

    proc.wait()

    if proc.returncode != 0:
        print(f"⚠️ 脚本异常退出，返回码: {proc.returncode}")

    print("---")

正在执行: /home/rice/notebook/skel/CSAD 模型（基于板块中地位）.py
使用解释器: /usr/local/envs/Python3.9/bin/python


In [3]:
[item for item in os.listdir('.') if ".py" in item]

['平均异常日收益率（ABNRETAVG）.py',
 '每日最大异常成交量（ABNVOLD）.py',
 '每日平均异常成交（ABNVOLAVG）.py',
 '平均换手率（TURNAVG）.py',
 'CSAD 模型（基于“小型”板块）.py',
 'CSAD 模型（基于板块中地位）.py',
 'LSV 模型.py',
 'SLSV 模型（对称 LSV 模型）.py',
 '基于 PE 的分析师锚定效应偏差（CAF_EP）.py',
 '基于市场波动率的注意力捕捉.py',
 '基于异常收益的注意力捕捉.py',
 '基于换手率的注意力捕捉.py',
 '基于沪深总成交额的注意力捕捉.py',
 '基于涨跌停的注意力捕捉.py',
 '基于换手率的注意力溢出（SPILL）.py',
 '衰减加权月频成交量（ATTN）.py',
 'CSAD 模型（基于时序回归变形）.py',
 'ABNRETD_因子检验.py',
 'ABNRETD_数据检查.py',
 '最大异常日收益率（ABNRETD）.py',
 '未实现盈利量（CGO）.py',
 '基于异常收益的注意力溢出（SPILL）.py',
 '前景价值（TK）.py',
 '基于 PB 的前景价值 TK.py']

In [ ]:
import os
from pathlib import Path
import pandas as pd
ignore = [".ipynb_checkpoints", "benchmark_data"]
[item for item in os.listdir('./factor_results') if item not in ignore]

In [10]:
factor_folder = [item for item in os.listdir('./factor_results') if item not in ignore]
for factor in factor_folder:
    save_dir = Path(f"factor_results/{factor}")
    ic = pd.read_csv(
        save_dir / "ic.csv",
        index_col="date",
        parse_dates=["date"],
    ).iloc[:, 0].dropna()
    print(f"{factor}: {ic.mean()}")

每日最大异常成交量（ABNVOLD）: 0.04358133401626941
平均异常日收益率（ABNRETAVG）: 0.06085570304219788
前景价值（TK）: -0.003318387434368924
基于 PB 的前景价值 TK: -0.0013550991776773685
LSV 模型: 0.006367479691169617
基于异常收益的注意力捕捉: 0.026899155112612785
CSAD 模型（基于时序回归变形）: 0.01427929514467051
未实现盈利量（CGO）: 0.03191226271986045
基于异常收益的注意力溢出（SPILL）: 0.04475829861278153
基于换手率的注意力捕捉: 0.029249844094572184
基于市场波动率的注意力捕捉: 0.03062714566250019
每日平均异常成交（ABNVOLAVG）: 0.04143068005286419
平均换手率（TURNAVG）: 0.043783984886482066
SLSV 模型（对称 LSV 模型）: -0.0015701674920779144
基于 PE 的分析师锚定效应偏差（CAF_EP）: 0.014255680179169242
CSAD 模型（基于“小型”板块）: 0.019646703673482587
CSAD 模型（基于板块中地位）: -0.031724281945735455
衰减加权月频成交量（ATTN）: 0.010209980531909383
最大异常日收益率（ABNRETD）: 0.056574783644962316
基于沪深总成交额的注意力捕捉: 0.032400973214168564
基于换手率的注意力溢出（SPILL）: 0.03781257631261429
基于涨跌停的注意力捕捉: 0.03192754020787116


In [24]:
from pathlib import Path
import pandas as pd
from rqdatac import get_price

BENCHMARK = "000985.XSHG"

benchmark_dir = Path("factor_results/benchmark_data")
benchmark_dir.mkdir(parents=True, exist_ok=True)

benchmark_file = benchmark_dir / "000985.XSHG_close.csv"

# 中证全指历史较早，2005 年开始保存即可覆盖常用研究区间
benchmark_price = get_price(
    BENCHMARK,
    start_date="2005-01-01",
    end_date=pd.Timestamp.today().strftime("%Y-%m-%d"),
    frequency="1d",
    fields="close",
    adjust_type="pre",
    expect_df=True,
)

benchmark_close_all = benchmark_price["close"]

# 兼容米筐返回 MultiIndex 的情形
if isinstance(benchmark_close_all.index, pd.MultiIndex):
    benchmark_close_all = (
        benchmark_close_all
        .unstack("order_book_id")
        .iloc[:, 0]
    )

benchmark_close_all.index = pd.to_datetime(benchmark_close_all.index)
benchmark_close_all = (
    benchmark_close_all
    .sort_index()
    .rename("close")
)

benchmark_close_all.to_csv(
    benchmark_file,
    index_label="date",
)

print("基准全历史数据已保存至：", benchmark_file.resolve())
print(
    "覆盖区间：",
    benchmark_close_all.index.min().date(),
    "至",
    benchmark_close_all.index.max().date(),
)

基准全历史数据已保存至： /home/rice/notebook/skel/factor_results/benchmark_data/000985.XSHG_close.csv
覆盖区间： 2005-01-04 至 2026-09-08


In [31]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import stats

TRADING_DAYS = 252
RISK_FREE_RATE = 0.0

# 修改为实际保存目录
save_dir = Path("factor_results/平均异常日收益率（ABNRETAVG）")

# ========= 1. 读取本地回测参数与 IC 数据 =========
with open(save_dir / "params.json", "r", encoding="utf-8") as f:
    params = json.load(f)

ic = pd.read_csv(
    save_dir / "ic.csv",
    index_col="date",
    parse_dates=["date"],
).iloc[:, 0].dropna()

ic.name = "rank_ic"

# ========= 2. IC 指标 =========
ic_t_stat, ic_p_value = stats.ttest_1samp(ic, 0)
ic_std = ic.std(ddof=1)

ic_metrics = pd.Series({
    "样本天数": len(ic),
    "平均 Rank IC": ic.mean(),
    "Rank IC 标准差": ic_std,
    "ICIR": ic.mean() / ic_std,
    "年化 ICIR": ic.mean() / ic_std * np.sqrt(TRADING_DAYS),
    "IC t 值": ic_t_stat,
    "IC p 值": ic_p_value,
    "正 IC 占比": (ic > 0).mean(),
    "负 IC 占比": (ic < 0).mean(),
    "IC 偏度": ic.skew(),
    "IC 峰度": ic.kurtosis(),
    "IC 一阶自相关": ic.autocorr(lag=1),
    "累计 Rank IC": ic.cumsum().iloc[-1],
})


def format_number(x):
    """普通数值保留 3 位小数。"""
    if pd.isna(x):
        return ""
    return f"{x:,.3f}"


ic_metrics_df = ic_metrics.to_frame("Result")

display(
    ic_metrics_df.style
    .format({"Result": format_number})
    .format(
        {"Result": "{:,.0f}"},
        subset=pd.IndexSlice[["样本天数"], ["Result"]],
    )
    .format(
        {"Result": "{:.3e}"},
        subset=pd.IndexSlice[["IC p 值"], ["Result"]],
    )
)

# ========= 3. 读取本地分组累计收益、换手率 =========
# CSV 原始格式：行=q0~q9；列=日期；值=累计收益率
quantile_factor_returns = pd.read_csv(
    save_dir / "quantile_factor_returns.csv",
    index_col=0,
).apply(pd.to_numeric, errors="coerce")

quantile_factor_returns.columns = pd.to_datetime(
    quantile_factor_returns.columns
)

quantile_turnover = pd.read_csv(
    save_dir / "quantile_turnover.csv",
    index_col=0,
).apply(pd.to_numeric, errors="coerce")

quantile_turnover.columns = pd.to_datetime(
    quantile_turnover.columns
)

# 转为：行=日期；列=q0~q9
group_cumret = quantile_factor_returns.T.sort_index()

# 累计收益率 -> 净值 -> 日收益率
group_nav = 1 + group_cumret

# 禁止 pandas 用前值填充缺失净值，避免把缺失日错误视为 0 收益
group_daily_ret = (
    group_nav
    .pct_change(fill_method=None)
    .dropna(how="all")
)

# 换手率转置并匹配收益率日期
group_turnover = (
    quantile_turnover.T
    .sort_index()
    .reindex(group_daily_ret.index)
)

# ========= 4. 从本地全量基准库中截取本次回测区间 =========
benchmark_file = Path(
    "factor_results/benchmark_data/000985.XSHG_close.csv"
)

benchmark_close_all = pd.read_csv(
    benchmark_file,
    index_col="date",
    parse_dates=["date"],
).iloc[:, 0]

benchmark_close_all = (
    pd.to_numeric(benchmark_close_all, errors="coerce")
    .dropna()
    .sort_index()
)

# 先对全历史价格计算收益率，再按本次策略日期截取。
# 这样首日收益率也会正确使用前一个交易日的收盘价。
benchmark_ret_all = benchmark_close_all.pct_change()

backtest_start = group_daily_ret.index.min()
backtest_end = group_daily_ret.index.max()

benchmark_ret = (
    benchmark_ret_all
    .loc[backtest_start:backtest_end]
    .reindex(group_daily_ret.index)
)

print(
    "本次绩效计算使用的基准区间：",
    benchmark_ret.index.min().date(),
    "至",
    benchmark_ret.index.max().date(),
)

# 这样不会在基准缺失时静默跳过日期，导致策略年化收益与基准年化收益的样本期不同。
benchmark_ret = benchmark_ret.reindex(group_daily_ret.index)

missing_benchmark_days = benchmark_ret.isna()

if missing_benchmark_days.any():
    missing_dates = benchmark_ret.index[missing_benchmark_days].strftime(
        "%Y-%m-%d"
    ).tolist()

    raise ValueError(
        "基准数据未覆盖以下策略交易日，无法进行严格的相对绩效计算："
        f"{missing_dates[:10]}"
    )

print(
    "策略与基准共同样本天数：",
    len(group_daily_ret.index),
)

# ========= 5. 单组合绩效函数 =========
def performance_metrics(daily_ret, benchmark_ret=None, annual_rf=0.0):
    daily_ret = daily_ret.dropna()

    if benchmark_ret is not None:
        pair = pd.concat(
            [
                daily_ret.rename("strategy"),
                benchmark_ret.rename("benchmark"),
            ],
            axis=1,
        ).dropna()

        if len(pair) < 2:
            return pd.Series(dtype=float)

        # 只要计算基准相关指标，所有策略指标也基于共同日期。
        daily_ret = pair["strategy"]
        benchmark_ret = pair["benchmark"]

    if len(daily_ret) < 2:
        return pd.Series(dtype=float)

    rf_daily = (1 + annual_rf) ** (1 / TRADING_DAYS) - 1
    excess_ret = daily_ret - rf_daily

    nav = (1 + daily_ret).cumprod()
    cumulative_return = nav.iloc[-1] - 1
    annual_return = nav.iloc[-1] ** (
        TRADING_DAYS / len(daily_ret)
    ) - 1

    daily_std = daily_ret.std(ddof=1)
    annual_volatility = daily_std * np.sqrt(TRADING_DAYS)

    sharpe = (
        excess_ret.mean() / excess_ret.std(ddof=1)
        * np.sqrt(TRADING_DAYS)
        if excess_ret.std(ddof=1) > 0 else np.nan
    )

    downside_volatility = (
        np.sqrt(np.mean(np.minimum(excess_ret, 0) ** 2))
        * np.sqrt(TRADING_DAYS)
    )

    sortino = (
        excess_ret.mean() * TRADING_DAYS / downside_volatility
        if downside_volatility > 0 else np.nan
    )

    drawdown = nav / nav.cummax() - 1
    max_drawdown = drawdown.min()

    calmar = (
        annual_return / abs(max_drawdown)
        if max_drawdown < 0 else np.nan
    )

    output = {
        "累计收益率": cumulative_return,
        "年化收益率": annual_return,
        "年化波动率": annual_volatility,
        "夏普比率": sharpe,
        "Sortino 比率": sortino,
        "最大回撤": max_drawdown,
        "Calmar 比率": calmar,
        "日胜率": (daily_ret > 0).mean(),
        "平均日收益": daily_ret.mean(),
        "日收益偏度": daily_ret.skew(),
        "日收益峰度": daily_ret.kurtosis(),
    }

    if benchmark_ret is not None:
        r = daily_ret
        b = benchmark_ret

        benchmark_annual_return = (
            (1 + b).prod() ** (TRADING_DAYS / len(b)) - 1
        )

        beta = (
            np.cov(r, b, ddof=1)[0, 1]
            / np.var(b, ddof=1)
        )

        alpha_daily = (
            (r - rf_daily).mean()
            - beta * (b - rf_daily).mean()
        )

        active_ret = r - b
        tracking_error = (
            active_ret.std(ddof=1) * np.sqrt(TRADING_DAYS)
        )

        information_ratio = (
            active_ret.mean() / active_ret.std(ddof=1)
            * np.sqrt(TRADING_DAYS)
            if active_ret.std(ddof=1) > 0 else np.nan
        )

        # 几何超额收益：策略净值相对基准净值的年化增速
        relative_nav = ((1 + r) / (1 + b)).cumprod()
        annual_geometric_excess = (
            relative_nav.iloc[-1]
            ** (TRADING_DAYS / len(relative_nav))
            - 1
        )

        output.update({
            "基准年化收益率": benchmark_annual_return,
            "年化超额收益（几何）": annual_geometric_excess,
            "年化收益差（算术）": (
                annual_return - benchmark_annual_return
            ),
            "Alpha（年化）": alpha_daily * TRADING_DAYS,
            "Beta": beta,
            "跟踪误差": tracking_error,
            "信息比率": information_ratio,
        })

    return pd.Series(output)

# ========= 6. 十个分组、total 与多空组合 =========
group_metrics = pd.DataFrame({
    group: performance_metrics(
        group_daily_ret[group],
        benchmark_ret,
        annual_rf=RISK_FREE_RATE,
    )
    for group in group_daily_ret.columns
}).T

# total：十个分组每日等权组合
total_daily_ret = group_daily_ret.mean(axis=1)
total_nav = (1 + total_daily_ret).cumprod()
total_cumret = total_nav - 1

TOTAL_GROUP = "total（十组等权）"

# 🚨严格的“全股票池等权组合”无法从十个分组的累计收益曲线完全还原，因为各组有效股票数、分组边界与缺失股票可能不同。
# 若要计算它，需要额外保存每日分组明细或逐股票收益数据。
group_metrics.loc[TOTAL_GROUP] = performance_metrics(
    total_daily_ret,
    benchmark_ret,
    annual_rf=RISK_FREE_RATE,
)

# 注意：确认 q0 是低因子值组、q9 是高因子值组后再使用
LONG_GROUP = "q0"
SHORT_GROUP = "q9"

long_short_daily_ret = (
    group_daily_ret[LONG_GROUP]
    - group_daily_ret[SHORT_GROUP]
)

# 多空策略本身通常不再与宽基指数比较
group_metrics.loc[f"long_short_{LONG_GROUP}_{SHORT_GROUP}"] = (
    performance_metrics(
        long_short_daily_ret,
        benchmark_ret=benchmark_ret,
        annual_rf=RISK_FREE_RATE,
    )
)

display(group_metrics.style.format(format_number))

# ========= 7. 换手率汇总 =========
turnover_metrics = pd.DataFrame({
    "平均日换手率": group_turnover.mean(),
    "中位数日换手率": group_turnover.median(),
    "年化换手率": group_turnover.mean() * TRADING_DAYS,
    "最大单日换手率": group_turnover.max(),
})

turnover_metrics.loc[TOTAL_GROUP] = {
    "平均日换手率": group_turnover.mean(axis=1).mean(),
    "中位数日换手率": group_turnover.mean(axis=1).median(),
    "年化换手率": group_turnover.mean(axis=1).mean() * TRADING_DAYS,
    "最大单日换手率": group_turnover.mean(axis=1).max(),
}

# q0 多头 + q9 空头。
# 与 long_short_daily_ret = q0 - q9 的收益定义一致：
# 两侧各持有 100% 名义本金，因此总换手率 = 两腿换手率之和。
long_short_turnover = group_turnover[
    [LONG_GROUP, SHORT_GROUP]
].sum(axis=1, min_count=2)

turnover_metrics.loc[
    f"long_short_{LONG_GROUP}_{SHORT_GROUP}"
] = {
    "平均日换手率": long_short_turnover.mean(),
    "中位数日换手率": long_short_turnover.median(),
    "年化换手率": long_short_turnover.mean() * TRADING_DAYS,
    "最大单日换手率": long_short_turnover.max(),
}

display(turnover_metrics.style.format(format_number))

,Result
样本天数,"1,454"
平均 Rank IC,-0.061
Rank IC 标准差,0.137
ICIR,-0.444
年化 ICIR,-7.041
IC t 值,-16.913
IC p 值,1.032e-58
正 IC 占比,0.332
负 IC 占比,0.668
IC 偏度,0.114


本次绩效计算使用的基准区间： 2020-01-06 至 2025-12-31
策略与基准共同样本天数： 1453


,累计收益率,年化收益率,年化波动率,夏普比率,Sortino 比率,最大回撤,Calmar 比率,日胜率,平均日收益,日收益偏度,日收益峰度,基准年化收益率,年化超额收益（几何）,年化收益差（算术）,Alpha（年化）,Beta,跟踪误差,信息比率
q0,2.695,0.254,0.224,1.125,1.551,-0.334,0.762,0.574,0.001,-0.828,10.270,0.046,0.200,0.209,0.192,0.931,0.125,1.497
q1,2.735,0.257,0.229,1.114,1.536,-0.312,0.823,0.570,0.001,-0.855,10.240,0.046,0.202,0.211,0.193,0.963,0.124,1.541
q2,2.524,0.244,0.232,1.060,1.465,-0.308,0.793,0.569,0.001,-0.766,10.197,0.046,0.190,0.199,0.181,0.994,0.118,1.530
q3,2.144,0.220,0.234,0.966,1.337,-0.307,0.715,0.562,0.001,-0.707,9.937,0.046,0.167,0.174,0.160,1.026,0.113,1.434
q4,1.887,0.202,0.237,0.897,1.238,-0.305,0.662,0.553,0.001,-0.717,9.498,0.046,0.149,0.156,0.144,1.057,0.106,1.391
q5,1.616,0.182,0.239,0.818,1.126,-0.301,0.604,0.554,0.001,-0.669,8.714,0.046,0.130,0.136,0.126,1.083,0.102,1.282
q6,1.323,0.157,0.246,0.718,0.989,-0.319,0.493,0.546,0.001,-0.651,8.788,0.046,0.107,0.112,0.104,1.124,0.102,1.100
q7,0.902,0.118,0.253,0.568,0.778,-0.371,0.318,0.535,0.001,-0.590,6.689,0.046,0.069,0.072,0.069,1.158,0.106,0.743
q8,-0.428,-0.092,0.283,-0.199,-0.266,-0.643,-0.144,0.515,-0.000,-0.459,4.863,0.046,-0.132,-0.138,-0.136,1.226,0.148,-0.817
q9,-0.587,-0.142,0.297,-0.367,-0.489,-0.745,-0.191,0.511,-0.000,-0.392,4.567,0.046,-0.180,-0.188,-0.191,1.263,0.164,-1.062


,平均日换手率,中位数日换手率,年化换手率,最大单日换手率
q0,0.101,0.096,25.558,0.375
q1,0.216,0.207,54.449,0.735
q2,0.253,0.240,63.659,0.837
q3,0.266,0.253,66.953,0.897
q4,0.266,0.253,67.134,0.872
q5,0.257,0.244,64.760,0.864
q6,0.239,0.229,60.275,0.822
q7,0.213,0.204,53.585,0.773
q8,0.133,0.131,33.445,0.550
q9,0.071,0.068,17.980,0.312


In [32]:
ic.mean()

-0.06085570304209929

- 算术年化：平均每天大约赚多少，乘以 252。
- 几何年化：真实复利后，净值每年实际增长多少。

例如连续两天收益：

| 日期 | 日收益 |
|---|---:|
| 第一天 | +10% |
| 第二天 | -10% |

算术平均日收益是：

$$
(10\% - 10\%) / 2 = 0
$$

看起来没有收益。

但实际净值是：

$$
1.10 \times 0.90 = 0.99
$$

两天实际亏损 1%。这就是波动拖累，也只有几何收益能反映出来。

对你的绩效指标：

- `年化收益率` 应使用几何年化，因为它描述资金曲线从起点到终点的真实复利增长。
- `年化收益差（算术）` 是“策略年化收益率 − 基准年化收益率”，便于直观比较两个组合的最终表现。
- `年化超额收益（几何）` 是策略净值相对基准净值的复利增长率：

$$
\left[
\prod_t \frac{1+r_{strategy,t}}{1+r_{benchmark,t}}
\right]^{252/T} - 1
$$

它衡量“持有策略相对于持有基准，财富每年多增长多少”。

- `Alpha（年化）` 通常采用算术年化：

$$
\alpha_{\text{annual}} \approx 252 \times \alpha_{\text{daily}}
$$

因为 Alpha 来自日收益回归的截距，是条件期望收益的线性估计量，不是一个可直接复利的单一投资组合净值。

因此，建议你的表格采用：

- 年化收益率：几何年化
- 年化超额收益（几何）：策略相对基准的真实复利优势
- 年化收益差（算术）：两个年化收益率直接相减
- Alpha（年化）：日 Alpha × 252

这样每个指标的财务含义明确，不会把“期望收益”“实际净值增长”和“回归截距”混为一谈。